# Spark et Databricks (Partie 5) : le Machine Learning avec MLlib

📖 **Le mini-cours complet** : [Spark et Databricks : le Machine Learning à grande échelle avec MLlib (Partie 5)](https://www.monshotdata.com/p/spark-databricks-machine-learning-mllib-partie-5)

🧭 **Le cours** : [Bonnes pratiques d'ingénierie › 1.4 Passage à l'échelle et calcul distribué](https://www.monshotdata.com/mini-cours/bonnes-pratiques-ingenierie#h2sc14)

---

MLlib est la bibliothèque de Machine Learning de Spark : elle entraîne des modèles sur des données qui ne tiennent pas dans une seule machine. Ici tu enchaînes régression linéaire, régression logistique et KMeans avec le même workflow, puis tu les industrialises avec des pipelines.

**À la fin de ce notebook, tu sauras :**

- regrouper tes features dans une colonne vecteur avec `VectorAssembler` ;
- entraîner, évaluer et utiliser une régression linéaire et une régression logistique ;
- mesurer n'importe quel modèle avec les évaluateurs de `pyspark.ml.evaluation` ;
- construire un pipeline, le régler par validation croisée, le sauvegarder et le recharger.

### Avant de lancer

Ce notebook tourne sur **Databricks**, pas sur Colab. Un compte **Databricks Free Edition** (gratuit) suffit.

- **Ouvrir le notebook** : dans Databricks, Workspace → Import, puis dépose ce fichier `.ipynb` (ou colle l'URL du fichier sur GitHub).
- **Le lancer** : attache-le au calcul **serverless**, puis exécute les cellules de haut en bas (ou « Run all »).
- **Les données** : la première cellule crée le Volume `workspace.default.cours_spark`, qui sert aux fichiers temporaires et au modèle sauvegardé. Les données sont générées dans le notebook.

🧪 *Testé le 21 novembre 2026 : exécution complète de haut en bas sur Databricks Free Edition (calcul serverless, Spark 4.2, Python 3.12). Les sorties affichées viennent de cette exécution ; les identifiants techniques (adresse de la session, noms de fichiers `part-…`, `uid` des modèles) changent à chaque lancement.*

## 1. Préparation

Ce notebook s'exécute seul : cette cellule crée le Volume utilisé pour les fichiers temporaires et le modèle sauvegardé.

In [1]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.cours_spark")

DataFrame[]


## 2. Représenter les features avec VectorAssembler

MLlib attend toutes les features dans une seule colonne vecteur.

In [2]:
# Représenter les features : VectorAssembler
from pyspark.ml.feature import VectorAssembler

df = spark.createDataFrame(
    [(1, 4.0, 7.5, 2.0), (0, 1.5, 3.2, 8.1), (1, 6.2, 9.0, 1.4)],
    ["label", "colA", "colB", "colC"],
)

assembleur = VectorAssembler(inputCols=["colA", "colB", "colC"],
                             outputCol="features")
df_assemble = assembleur.transform(df)
df_assemble.show()
# +-----+----+----+----+-------------+
# |label|colA|colB|colC|     features|
# +-----+----+----+----+-------------+
# |    1| 4.0| 7.5| 2.0|[4.0,7.5,2.0]|
# |    0| 1.5| 3.2| 8.1|[1.5,3.2,8.1]|
# |    1| 6.2| 9.0| 1.4|[6.2,9.0,1.4]|
# +-----+----+----+----+-------------+

+-----+----+----+----+-------------+
|label|colA|colB|colC|     features|
+-----+----+----+----+-------------+
|    1| 4.0| 7.5| 2.0|[4.0,7.5,2.0]|
|    0| 1.5| 3.2| 8.1|[1.5,3.2,8.1]|
|    1| 6.2| 9.0| 1.4|[6.2,9.0,1.4]|
+-----+----+----+----+-------------+



In [3]:
# Garder seulement les colonnes utiles au modèle
donnees = df_assemble.select("features", "label")
donnees.show()
# +-------------+-----+
# |     features|label|
# +-------------+-----+
# |[4.0,7.5,2.0]|    1|
# |[1.5,3.2,8.1]|    0|
# |[6.2,9.0,1.4]|    1|
# +-------------+-----+
donnees.printSchema()
# root
#  |-- features: vectorudt (nullable = true)
#  |-- label: long (nullable = true)

+-------------+-----+
|     features|label|
+-------------+-----+
|[4.0,7.5,2.0]|    1|
|[1.5,3.2,8.1]|    0|
|[6.2,9.0,1.4]|    1|
+-------------+-----+

root
 |-- features: vectorudt (nullable = true)
 |-- label: long (nullable = true)


## 3. Régression linéaire

Dataset synthétique, assemblage, découpage train/test, entraînement, évaluation et prédiction.

In [4]:
# Un dataset de régression synthétique avec scikit-learn
import pandas as pd
from sklearn.datasets import make_regression

X, y, vrais_coefs = make_regression(n_samples=10_000, n_features=5, noise=10,
                                    coef=True, random_state=42)

colonnes = [f"f_{i}" for i in range(1, 6)]
df_pandas = pd.DataFrame(X, columns=colonnes)
df_pandas["label"] = y

df_reg = spark.createDataFrame(df_pandas)
df_reg.select("f_1", "f_2", "label").show(3)
# +--------------------+--------------------+-------------------+
# |                 f_1|                 f_2|              label|
# +--------------------+--------------------+-------------------+
# |0.015983766722198454|-0.25042721743676133| -70.45555225488955|
# |  0.8248201644828588| -1.1838730900058776|-11.633550657674068|
# | -1.5602419775537173|  1.8546621338595848| 3.4222273209101832|
# +--------------------+--------------------+-------------------+
# only showing top 3 rows
print("Vrais coefficients :", vrais_coefs.round(3))
# Vrais coefficients : [91.405 94.775  0.277 38.252  5.356]

+--------------------+--------------------+-------------------+
|                 f_1|                 f_2|              label|
+--------------------+--------------------+-------------------+
|0.015983766722198454|-0.25042721743676133| -70.45555225488955|
|  0.8248201644828588| -1.1838730900058776|-11.633550657674068|
| -1.5602419775537173|  1.8546621338595848| 3.4222273209101832|
+--------------------+--------------------+-------------------+
only showing top 3 rows
Vrais coefficients : [91.405 94.775  0.277 38.252  5.356]


In [5]:
# Assembler les 5 features dans une seule colonne
assembleur = VectorAssembler(inputCols=colonnes, outputCol="features")
donnees_reg = assembleur.transform(df_reg).select("features", "label")

# Découper en train (80 %) et test (20 %),
# avec une graine pour la reproductibilité
train, test = donnees_reg.randomSplit([0.8, 0.2], seed=42)
print(train.count(), test.count())
# 8046 1954

8046 1954


In [6]:
# Entraîner une régression linéaire
from pyspark.ml.regression import LinearRegression

reg_lin = LinearRegression(featuresCol="features", labelCol="label",
                           predictionCol="prediction")
modele = reg_lin.fit(train)

print("Coefficients :", modele.coefficients)
# Coefficients : [91.49904486823432,94.76980333699721,0.31402090935759985,
#                 38.291131543445296,5.428350306739271]
print("Intercept :", round(modele.intercept, 4))
# Intercept : -0.0424

Coefficients : [91.49904486823432,94.76980333699721,0.31402090935759985,38.291131543445296,5.428350306739271]
Intercept : -0.0424


In [7]:
# Les métriques sur le train set
resume = modele.summary
print("R² :", round(resume.r2, 4))
# R² : 0.9946
print("R² ajusté :", round(resume.r2adj, 4))
# R² ajusté : 0.9946
print("MAE :", round(resume.meanAbsoluteError, 3))
# MAE : 8.057
print("RMSE :", round(resume.rootMeanSquaredError, 3))
# RMSE : 10.118
print("MSE :", round(resume.meanSquaredError, 3))
# MSE : 102.378

resume.residuals.show(3)
# +-------------------+
# |          residuals|
# +-------------------+
# |-10.612293490187398|
# |  11.35131549134124|
# |  6.501880886067919|
# +-------------------+
# only showing top 3 rows
resume.predictions.show(3)
# +--------------------+-------------------+-------------------+
# |            features|              label|         prediction|
# +--------------------+-------------------+-------------------+
# |[-2.9659557581441...| -228.8412098111578| -218.2289163209704|
# |[-2.8876138501461...|-168.58820488004963|-179.93952037139087|
# |[-2.7373370562130...| 31.949890846087026| 25.448009960019107|
# +--------------------+-------------------+-------------------+
# only showing top 3 rows

R² : 0.9946
R² ajusté : 0.9946
MAE : 8.057
RMSE : 10.118
MSE : 102.378
+-------------------+
|          residuals|
+-------------------+
|-10.612293490187398|
|  11.35131549134124|
|  6.501880886067919|
+-------------------+
only showing top 3 rows
+--------------------+-------------------+-------------------+
|            features|              label|         prediction|
+--------------------+-------------------+-------------------+
|[-2.9659557581441...| -228.8412098111578| -218.2289163209704|
|[-2.8876138501461...|-168.58820488004963|-179.93952037139087|
|[-2.7373370562130...| 31.949890846087026| 25.448009960019107|
+--------------------+-------------------+-------------------+
only showing top 3 rows


In [8]:
# Évaluer sur le test set : evaluate() a besoin des labels
resultats_test = modele.evaluate(test)
print("R² (test) :", round(resultats_test.r2, 4))
# R² (test) : 0.9944
print("RMSE (test) :", round(resultats_test.rootMeanSquaredError, 3))
# RMSE (test) : 10.342

R² (test) : 0.9944
RMSE (test) : 10.342


In [9]:
# Prédire sur des données sans label : transform()
sans_label = test.select("features")
modele.transform(sans_label).show(3)
# +--------------------+-------------------+
# |            features|         prediction|
# +--------------------+-------------------+
# |[-2.8562754133411...|-272.47598070115833|
# |[-2.5632897868965...| -246.9121023803463|
# |[-2.4237473182779...| -138.8512023890343|
# +--------------------+-------------------+
# only showing top 3 rows

+--------------------+-------------------+
|            features|         prediction|
+--------------------+-------------------+
|[-2.8562754133411...|-272.47598070115833|
|[-2.5632897868965...| -246.9121023803463|
|[-2.4237473182779...| -138.8512023890343|
+--------------------+-------------------+
only showing top 3 rows


## 4. Régression logistique et évaluateurs

Classification binaire, métriques du résumé et évaluateurs de `pyspark.ml.evaluation`.

In [10]:
# Un dataset de classification binaire synthétique
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=10_000, n_features=5, n_informative=3,
                           n_redundant=1, random_state=42)
df_pandas = pd.DataFrame(X, columns=colonnes)
df_pandas["label"] = y

df_clf = spark.createDataFrame(df_pandas)
donnees_clf = assembleur.transform(df_clf).select("features", "label")
train, test = donnees_clf.randomSplit([0.8, 0.2], seed=42)
train.groupBy("label").count().orderBy("label").show()
# +-----+-----+
# |label|count|
# +-----+-----+
# |    0| 4049|
# |    1| 3997|
# +-----+-----+

+-----+-----+
|label|count|
+-----+-----+
|    0| 4049|
|    1| 3997|
+-----+-----+



In [11]:
# Entraîner une régression logistique
from pyspark.ml.classification import LogisticRegression

reg_log = LogisticRegression(featuresCol="features", labelCol="label",
                             predictionCol="prediction")
modele = reg_log.fit(train)

print("Coefficients :", modele.coefficients)
# Coefficients : [1.0431315870523425,0.7254755464499193,1.5127498442281957,
#                 -3.3206596305090175,-0.010368806732106708]
print("Intercept :", round(modele.intercept, 4))
# Intercept : -1.1183

Coefficients : [1.0431315870523425,0.7254755464499193,1.5127498442281957,-3.3206596305090175,-0.010368806732106708]
Intercept : -1.1183


In [12]:
# Les métriques disponibles directement sur le train set
resume = modele.summary
print("Accuracy :", round(resume.accuracy, 4))
# Accuracy : 0.8706
print("AUC ROC :", round(resume.areaUnderROC, 4))
# AUC ROC : 0.9427

Accuracy : 0.8706
AUC ROC : 0.9427


In [13]:
# Toutes les colonnes de sortie du modèle, sur le test set
predictions = modele.transform(test)
predictions.select("rawPrediction", "probability", "prediction", "label").show(3)
# +--------------------+--------------------+----------+-----+
# |       rawPrediction|         probability|prediction|label|
# +--------------------+--------------------+----------+-----+
# |[5.04691150885668...|[0.99361191082644...|       0.0|    0|
# |[2.97321296809100...|[0.95134920152735...|       0.0|    0|
# |[6.80335740676727...|[0.99889118880507...|       0.0|    0|
# +--------------------+--------------------+----------+-----+
# only showing top 3 rows

+--------------------+--------------------+----------+-----+
|       rawPrediction|         probability|prediction|label|
+--------------------+--------------------+----------+-----+
|[5.04691150885668...|[0.99361191082644...|       0.0|    0|
|[2.97321296809100...|[0.95134920152735...|       0.0|    0|
|[6.80335740676727...|[0.99889118880507...|       0.0|    0|
+--------------------+--------------------+----------+-----+
only showing top 3 rows


In [14]:
# Les évaluateurs : n'importe quelle métrique, sur n'importe quel jeu
from pyspark.ml.evaluation import (BinaryClassificationEvaluator,
                                   MulticlassClassificationEvaluator)

eval_binaire = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction",
                                             labelCol="label")
eval_multi = MulticlassClassificationEvaluator(predictionCol="prediction",
                                               labelCol="label")

print("AUC ROC (test) :", round(eval_binaire.evaluate(predictions), 4))
auc_pr = eval_binaire.evaluate(predictions,
                               {eval_binaire.metricName: "areaUnderPR"})
print("AUC PR (test) :", round(auc_pr, 4))
for metrique in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    valeur = eval_multi.evaluate(predictions, {eval_multi.metricName: metrique})
    print(f"{metrique} (test) :", round(valeur, 4))

# AUC ROC (test) : 0.9435
# AUC PR (test) : 0.944
# accuracy (test) : 0.868
# f1 (test) : 0.868
# weightedPrecision (test) : 0.868
# weightedRecall (test) : 0.868

AUC ROC (test) : 0.9435
AUC PR (test) : 0.944
accuracy (test) : 0.868
f1 (test) : 0.868
weightedPrecision (test) : 0.868
weightedRecall (test) : 0.868


## 5. Le même workflow pour tous les algorithmes

Le même enchaînement `VectorAssembler`, `fit()`, `transform()` avec KMeans.

In [15]:
# Même workflow pour un autre algorithme : KMeans
from sklearn.datasets import make_blobs
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

X, _ = make_blobs(n_samples=1_000, centers=3, n_features=2, random_state=42)
df_points = spark.createDataFrame(pd.DataFrame(X, columns=["x", "y"]))
points = VectorAssembler(inputCols=["x", "y"],
                         outputCol="features").transform(df_points)

kmeans = KMeans(k=3, seed=42)
modele_km = kmeans.fit(points)
groupes = modele_km.transform(points)

print("Silhouette :", round(ClusteringEvaluator().evaluate(groupes), 4))
for centre in modele_km.clusterCenters():
    print("Centre :", centre.round(2))
groupes.groupBy("prediction").count().orderBy("prediction").show()

# Silhouette : 0.9678
# Centre : [-2.52  9.  ]
# Centre : [-6.87 -6.79]
# Centre : [4.73 2.06]
# +----------+-----+
# |prediction|count|
# +----------+-----+
# |         0|  334|
# |         1|  333|
# |         2|  333|
# +----------+-----+

Silhouette : 0.9678
Centre : [-2.52  9.  ]
Centre : [-6.87 -6.79]
Centre : [4.73 2.06]
+----------+-----+
|prediction|count|
+----------+-----+
|         0|  334|
|         1|  333|
|         2|  333|
+----------+-----+



## 6. Les pipelines

Pipeline, prédiction, métriques scikit-learn, validation croisée, sauvegarde et rechargement.

In [16]:
# Un pipeline : assemblage des features puis forêt aléatoire
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier

assembleur = VectorAssembler(inputCols=colonnes, outputCol="features")
foret = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)
pipeline = Pipeline(stages=[assembleur, foret])

train_brut, test_brut = df_clf.randomSplit([0.8, 0.2], seed=42)
modele_pipeline = pipeline.fit(train_brut)
print(modele_pipeline.stages)
# [VectorAssembler_ce424f4909c3,
#  RandomForestClassificationModel: uid=RandomForestClassifier_bffaba810948,
#  numTrees=20, numClasses=2, numFeatures=5]

[VectorAssembler_69e313210da8, RandomForestClassificationModel: uid=RandomForestClassifier_1f5f7beb1501, numTrees=20, numClasses=2, numFeatures=5]


In [17]:
# Prédire et évaluer : le pipeline assemble les features tout seul
predictions = modele_pipeline.transform(test_brut)
predictions.select("f_1", "f_2", "label", "probability", "prediction").show(3)
# +------------------+-------------------+-----+--------------------+----------+
# |               f_1|                f_2|label|         probability|prediction|
# +------------------+-------------------+-----+--------------------+----------+
# |-2.895157998538836|-3.3204870507247723|    0|[0.93318585907507...|       0.0|
# |-2.827222686253645|-2.1702740288811846|    0|[0.90022370711656...|       0.0|
# |-2.663819068607257| -1.613991375390427|    0|[0.96410776688612...|       0.0|
# +------------------+-------------------+-----+--------------------+----------+
# only showing top 3 rows

evaluateur = MulticlassClassificationEvaluator(labelCol="label",
                                               predictionCol="prediction",
                                               metricName="accuracy")
print("Accuracy (test) :", round(evaluateur.evaluate(predictions), 4))
# Accuracy (test) : 0.9013

+------------------+-------------------+-----+--------------------+----------+
|               f_1|                f_2|label|         probability|prediction|
+------------------+-------------------+-----+--------------------+----------+
|-2.895157998538836|-3.3204870507247723|    0|[0.93318585907507...|       0.0|
|-2.827222686253645|-2.1702740288811846|    0|[0.90022370711656...|       0.0|
|-2.663819068607257| -1.613991375390427|    0|[0.96410776688612...|       0.0|
+------------------+-------------------+-----+--------------------+----------+
only showing top 3 rows
Accuracy (test) : 0.9013


In [18]:
# Utiliser les métriques de scikit-learn sur un petit résultat
from sklearn.metrics import classification_report

lignes = predictions.select("label", "prediction").collect()
y_vrai = [ligne["label"] for ligne in lignes]
y_pred = [ligne["prediction"] for ligne in lignes]
print(classification_report(y_vrai, y_pred, digits=3))

#               precision    recall  f1-score   support
#
#            0      0.893     0.907     0.900       994
#            1      0.910     0.895     0.902      1033
#
#     accuracy                          0.901      2027
#    macro avg      0.901     0.901     0.901      2027
# weighted avg      0.901     0.901     0.901      2027

              precision    recall  f1-score   support

           0      0.893     0.907     0.900       994
           1      0.910     0.895     0.902      1033

    accuracy                          0.901      2027
   macro avg      0.901     0.901     0.901      2027
weighted avg      0.901     0.901     0.901      2027


In [19]:
# Bonus : chercher les meilleurs hyperparamètres par validation croisée
import os
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Sur Databricks serverless, CrossValidator a besoin d'un dossier temporaire
# dans un volume
os.environ["SPARKML_TEMP_DFS_PATH"] = (
    "/Volumes/workspace/default/cours_spark/tmp_ml")
os.makedirs(os.environ["SPARKML_TEMP_DFS_PATH"], exist_ok=True)

grille = (ParamGridBuilder()
          .addGrid(foret.numTrees, [20, 50])
          .addGrid(foret.maxDepth, [3, 6])
          .build())

validation = CrossValidator(estimator=pipeline, estimatorParamMaps=grille,
                            evaluator=evaluateur, numFolds=3, seed=42)
modele_cv = validation.fit(train_brut)

for params, score in zip(grille, modele_cv.avgMetrics):
    print({p.name: v for p, v in params.items()},
          "accuracy moyenne :", round(score, 4))

meilleur = modele_cv.bestModel
score_test = evaluateur.evaluate(meilleur.transform(test_brut))
print("Accuracy (test) du meilleur modèle :", round(score_test, 4))

# {'numTrees': 20, 'maxDepth': 3} accuracy moyenne : 0.8798
# {'numTrees': 20, 'maxDepth': 6} accuracy moyenne : 0.9218
# {'numTrees': 50, 'maxDepth': 3} accuracy moyenne : 0.8799
# {'numTrees': 50, 'maxDepth': 6} accuracy moyenne : 0.9228
# Accuracy (test) du meilleur modèle : 0.9235

{'numTrees': 20, 'maxDepth': 3} accuracy moyenne : 0.8798
{'numTrees': 20, 'maxDepth': 6} accuracy moyenne : 0.9218
{'numTrees': 50, 'maxDepth': 3} accuracy moyenne : 0.8799
{'numTrees': 50, 'maxDepth': 6} accuracy moyenne : 0.9228
Accuracy (test) du meilleur modèle : 0.9235


In [20]:
# Bonus : sauvegarder le pipeline entraîné, puis le recharger
from pyspark.ml import PipelineModel

chemin_modele = "/Volumes/workspace/default/cours_spark/modele_foret"
meilleur.write().overwrite().save(chemin_modele)

modele_recharge = PipelineModel.load(chemin_modele)
print("Accuracy (test) du modèle rechargé :",
      round(evaluateur.evaluate(modele_recharge.transform(test_brut)), 4))
# Accuracy (test) du modèle rechargé : 0.9235

Accuracy (test) du modèle rechargé : 0.9235


## 7. Ce qu'il faut retenir

- Utilise MLlib via `pyspark.ml`, l'API fondée sur les DataFrames ; `pyspark.mllib`, fondée sur les RDD, est en mode maintenance.
- Toutes les features doivent être regroupées dans une seule colonne vecteur, avec `VectorAssembler`.
- `evaluate()` mesure un modèle sur des données labellisées ; `transform()` prédit sur n'importe quelles données.
- Les évaluateurs (`BinaryClassificationEvaluator`, `MulticlassClassificationEvaluator`...) calculent toutes les métriques, pour n'importe quel modèle et sur n'importe quel jeu.
- Un pipeline enchaîne Transformers et Estimators : il se règle avec `CrossValidator` et se réutilise tel quel après `save()` et `load()`.

---

## Pour aller plus loin

- [Spark et Databricks : agrégations, jointures et fonctions de fenêtre en PySpark (Partie 4)](https://www.monshotdata.com/p/spark-databricks-agregations-jointures-fenetres-partie-4) — la partie précédente : groupBy, pivot, jointures, fonctions de fenêtre et valeurs manquantes
- [Spark et Databricks : du big data à ton premier notebook PySpark (Partie 1)](https://www.monshotdata.com/p/spark-databricks-calcul-distribue-partie-1) — le début du mini-cours, si tu veux tout reprendre : Big data, calcul distribué, RDD et premier notebook PySpark